In [1]:
# --- GPU / runtime check ---
import torch, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NO GPU")
assert torch.cuda.is_available(), "No CUDA GPU. Runtime > Change runtime type > A100 or L4."
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}  |  {vram:.1f} GB VRAM  |  CUDA {torch.version.cuda}")
assert vram > 20, f"{vram:.0f} GB is too small for Qwen3-8B in bf16 (~16.4 GB weights). Need L4/A100."
print("OK: this GPU can hold Qwen3-8B unquantised.")

NVIDIA L4, 23034 MiB
NVIDIA L4  |  23.7 GB VRAM  |  CUDA 12.8
OK: this GPU can hold Qwen3-8B unquantised.


In [2]:
# --- Download Qwen3-8B weights (bf16, no quantisation) ---
# Grabs the full repo to local disk; does NOT load into VRAM yet.
!pip -q install -U "huggingface_hub[hf_transfer]" >/dev/null
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # faster parallel download

from huggingface_hub import snapshot_download
path = snapshot_download(
    repo_id="Qwen/Qwen3-8B",
    allow_patterns=["*.safetensors", "*.json", "*.txt", "tokenizer*", "*.model"],
)
print("Downloaded to:", path)
!du -sh {path}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:310: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Downloaded to: /root/.cache/huggingface/hub/models--Qwen--Qwen3-8B/snapshots/b968826d9c46dd6066d109eabc6255188de91218
28K	/root/.cache/huggingface/hub/models--Qwen--Qwen3-8B/snapshots/b968826d9c46dd6066d109eabc6255188de91218


In [3]:
# --- Verify the weights actually downloaded ---
import glob, os
files = sorted(glob.glob(os.path.join(path, "*")))
for f in files:
    real = os.path.realpath(f)
    size = os.path.getsize(real) / 1e9
    print(f"{os.path.basename(f):45s} {size:7.2f} GB  {'(symlink)' if os.path.islink(f) else ''}")
total = sum(os.path.getsize(os.path.realpath(f)) for f in files) / 1e9
print(f"\nTOTAL real bytes: {total:.2f} GB")
# safetensors count
st = [f for f in files if f.endswith('.safetensors')]
print(f"safetensors shards: {len(st)}")

config.json                                      0.00 GB  (symlink)
generation_config.json                           0.00 GB  (symlink)
merges.txt                                       0.00 GB  (symlink)
model-00001-of-00005.safetensors                 4.00 GB  (symlink)
model-00002-of-00005.safetensors                 3.99 GB  (symlink)
model-00003-of-00005.safetensors                 3.96 GB  (symlink)
model-00004-of-00005.safetensors                 3.19 GB  (symlink)
model-00005-of-00005.safetensors                 1.24 GB  (symlink)
model.safetensors.index.json                     0.00 GB  (symlink)
tokenizer.json                                   0.01 GB  (symlink)
tokenizer_config.json                            0.00 GB  (symlink)
vocab.json                                       0.00 GB  (symlink)

TOTAL real bytes: 16.40 GB
safetensors shards: 5


In [4]:
# --- Load Qwen3-8B onto the L4 in bf16 ---
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

t0 = time.time()
tok = AutoTokenizer.from_pretrained(path)
model = AutoModelForCausalLM.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)
model.eval()
print(f"Loaded in {time.time()-t0:.0f}s")
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB / 23.7 GB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Loaded in 7s
VRAM allocated: 16.4 GB / 23.7 GB


In [5]:
# --- Smoke test: 6 common everyday queries ---
import torch, time

queries = [
    "What shall I do today?",
    "What's a quick and healthy dinner I can make tonight?",
    "Give me a 3-sentence pep talk for a Monday morning.",
    "Explain how a rainbow forms, simply.",
    "Suggest a good book to read this weekend and why.",
    "What are three tips to sleep better?",
]

for i, q in enumerate(queries, 1):
    msgs = [{"role": "user", "content": q}]
    text = tok.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,   # keep smoke test fast/short
    )
    inputs = tok(text, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=180, do_sample=True,
                             temperature=0.7, top_p=0.8)
    gen = out[0][inputs.input_ids.shape[1]:]
    ans = tok.decode(gen, skip_special_tokens=True).strip()
    dt = time.time() - t0
    ntok = gen.shape[0]
    print(f"\n{'='*70}\n[{i}] {q}   ({ntok} tok, {dt:.1f}s, {ntok/dt:.1f} tok/s)\n{'-'*70}\n{ans}")

print(f"\n{'='*70}\nPeak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")


[1] What shall I do today?   (180 tok, 13.2s, 13.7 tok/s)
----------------------------------------------------------------------
That's a great question! What would you like to do today? Here are a few suggestions to help you decide:

### 🌞 Morning Activities
- **Wake up early** and enjoy a peaceful morning with some meditation, yoga, or a quiet walk.
- **Make a healthy breakfast** – maybe try a new recipe or experiment with something different.
- **Read a book** or listen to an audiobook – pick something you've been wanting to read.

### 📚 Learning & Growth
- **Learn something new** – take an online course, watch a documentary, or read a book on a topic you're curious about.
- **Practice a skill** – whether it's playing an instrument, writing, coding, or drawing, set aside some time to improve.

### 🌿 Self-Care
- **Take a bath or shower** with some calming music or essential

[2] What's a quick and healthy dinner I can make tonight?   (180 tok, 11.9s, 15.1 tok/s)
--------------------

In [9]:
# === GCG setup: maximise mean first-token Shannon entropy, penalise outliers ===
# Adversarial block placed as a PREFIX (before the query text).
import math, torch
import torch.nn.functional as F
torch.manual_seed(0)
LN2 = math.log(2.0)

# freeze model -> only the prefix one-hot will carry gradients (saves memory)
for p in model.parameters():
    p.requires_grad_(False)

embed = model.get_input_embeddings()
W = embed.weight                       # [V, d]  (bf16)
V, d = W.shape
DEV = model.device
DT  = W.dtype

queries = [
    "What shall I do today?",
    "What's a quick and healthy dinner I can make tonight?",
    "Give me a 3-sentence pep talk for a Monday morning.",
    "Explain how a rainbow forms, simply.",
    "Suggest a good book to read this weekend and why.",
    "What are three tips to sleep better?",
]

# Split each templated prompt around the optimisable PREFIX -> (before_ids, after_ids)
# before = template header (query-independent) ; after = " " + query + assistant header
PRE, POST = [], []
for q in queries:
    templ = tok.apply_chat_template(
        [{"role": "user", "content": "{optim_str} " + q}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    before, after = templ.split("{optim_str}")
    b = tok(before, return_tensors="pt").input_ids.to(DEV)
    a = tok(after, add_special_tokens=False, return_tensors="pt").input_ids.to(DEV)
    PRE.append(embed(b))               # [1, Lb, d]
    POST.append(embed(a))              # [1, La, d]

N_OPT = 15
ALPHA = 1.0                            # outlier (std) penalty weight

@torch.no_grad()
def per_query_entropy(optim_ids):
    """optim_ids: [B, N_OPT] -> H bits [6, B]  (first-token Shannon entropy)."""
    B = optim_ids.shape[0]
    e_opt = embed(optim_ids)                                   # [B, N_OPT, d]
    Hs = []
    for pe, po in zip(PRE, POST):
        full = torch.cat([pe.expand(B, -1, -1), e_opt, po.expand(B, -1, -1)], dim=1)
        h_last = model.model(inputs_embeds=full).last_hidden_state[:, -1, :]
        logp = F.log_softmax(model.lm_head(h_last).float(), dim=-1)
        H = -(logp.exp() * logp).sum(-1) / LN2                 # bits [B]
        Hs.append(H)
    return torch.stack(Hs, 0)                                  # [6, B]

def objective(H):                       # H: [6, B] -> [B]
    return H.mean(0) - ALPHA * H.std(0, unbiased=False)

def suffix_grad(optim_ids):
    """Gradient of loss(=-objective) w.r.t. prefix one-hot -> [N_OPT, V]."""
    oh = torch.zeros(N_OPT, V, device=DEV, dtype=DT)
    oh.scatter_(1, optim_ids[0].unsqueeze(1), 1.0)
    oh.requires_grad_(True)
    e_opt = (oh @ W).unsqueeze(0)                              # [1, N_OPT, d]
    Hs = []
    for pe, po in zip(PRE, POST):
        full = torch.cat([pe, e_opt, po], dim=1)
        h_last = model.model(inputs_embeds=full).last_hidden_state[:, -1, :]
        logp = F.log_softmax(model.lm_head(h_last).float(), dim=-1)
        Hs.append((-(logp.exp() * logp).sum(-1) / LN2).squeeze(0))
    H = torch.stack(Hs, 0).unsqueeze(1)                        # [6,1]
    loss = -objective(H).sum()
    loss.backward()
    return oh.grad.detach()

# --- baselines ---
V_max = math.log2(V)
excl = tok(" !", add_special_tokens=False).input_ids[-1]
init_ids = torch.full((1, N_OPT), excl, device=DEV, dtype=torch.long)

H0 = per_query_entropy(init_ids)
print(f"Vocab={V}, max possible entropy = log2(V) = {V_max:.2f} bits")
print(f"Init prefix '! ...':  mean={H0.mean():.3f}  std={H0.std(unbiased=False):.3f}  "
      f"min={H0.min():.3f}  obj={objective(H0).item():.3f} bits")
print("  per-query:", [f"{x:.2f}" for x in H0.squeeze(1).tolist()])

Vocab=151936, max possible entropy = log2(V) = 17.21 bits
Init prefix '! ...':  mean=1.092  std=0.716  min=0.006  obj=0.375 bits
  per-query: ['1.44', '1.19', '0.01', '0.45', '2.25', '1.21']


In [7]:
# === GCG search loop ===
import time

def gcg(steps, search_width=48, topk=256, seed=0, verbose_every=10):
    torch.manual_seed(seed)
    ids = init_ids.clone()
    best_H = per_query_entropy(ids)
    best_obj = objective(best_H).item()
    best_ids = ids.clone()
    t0 = time.time()
    for step in range(1, steps + 1):
        grad = suffix_grad(ids)                       # [N_OPT, V]
        cand_tok = (-grad).topk(topk, dim=1).indices  # [N_OPT, topk] lower-loss swaps
        # build candidate batch: each = current suffix with one position swapped
        cand = ids.repeat(search_width, 1)
        pos = torch.randint(0, N_OPT, (search_width,), device=DEV)
        pick = torch.randint(0, topk, (search_width,), device=DEV)
        cand[torch.arange(search_width, device=DEV), pos] = cand_tok[pos, pick]
        cand = torch.cat([ids, cand], dim=0)          # keep current as a candidate
        H = per_query_entropy(cand)                   # [6, B+1]
        obj = objective(H)                            # [B+1]
        j = int(obj.argmax())
        if obj[j].item() > best_obj:
            best_obj = obj[j].item(); best_ids = cand[j:j+1].clone(); best_H = H[:, j:j+1]
        ids = cand[j:j+1].clone()                     # greedy move
        if step % verbose_every == 0 or step == steps:
            print(f"step {step:3d}  obj={best_obj:.3f}  mean={best_H.mean():.3f}  "
                  f"std={best_H.std(unbiased=False):.3f}  min={best_H.min():.3f}  "
                  f"({(time.time()-t0)/step:.2f}s/step)")
    return best_ids, best_obj, best_H

# quick 3-step validation (correctness + memory)
_ids, _obj, _H = gcg(steps=3, verbose_every=1)
print("free VRAM:", f"{(torch.cuda.mem_get_info()[0])/1e9:.1f} GB")

step   1  obj=1.300  mean=2.440  std=1.140  min=0.369  (4.86s/step)
step   2  obj=1.617  mean=2.160  std=0.543  min=1.395  (4.83s/step)
step   3  obj=1.723  mean=2.665  std=0.942  min=1.537  (4.83s/step)
free VRAM: 4.9 GB


In [10]:
# === Full GCG run (prefix) ===
best_ids, best_obj, best_H = gcg(steps=60, search_width=48, topk=256, verbose_every=10)

prefix_str = tok.decode(best_ids[0])
print("\n" + "="*70)
print("BEST 15-token prefix (ids):", best_ids[0].tolist())
print("BEST prefix (decoded)     :", repr(prefix_str))
print("-"*70)
print(f"objective (mean - {ALPHA}*std) : {best_obj:.3f} bits   (baseline {objective(H0).item():.3f})")
print(f"mean entropy : {best_H.mean():.3f} bits   (baseline {H0.mean():.3f},  max {V_max:.2f})")
print(f"std          : {best_H.std(unbiased=False):.3f}   min : {best_H.min():.3f}")
for q, h in zip(queries, best_H.squeeze(1).tolist()):
    print(f"   {h:5.2f} bits  |  {q}")

step  10  obj=4.189  mean=5.174  std=0.985  min=3.205  (4.58s/step)
step  20  obj=7.235  mean=7.500  std=0.265  min=7.094  (4.54s/step)
step  30  obj=7.310  mean=7.579  std=0.269  min=7.308  (4.54s/step)
step  40  obj=7.452  mean=7.498  std=0.045  min=7.431  (4.53s/step)
step  50  obj=7.658  mean=7.912  std=0.254  min=7.663  (4.53s/step)
step  60  obj=8.468  mean=8.887  std=0.420  min=8.127  (4.53s/step)

BEST 15-token prefix (ids): [41405, 144091, 99944, 753, 16787, 65810, 61804, 112618, 129259, 23645, 21732, 86214, 144069, 103572, 753]
BEST prefix (decoded)     : "=======\n딩予 !Private-------------\n �平板ちょっと nemyour?'\n\n▶屑 !"
----------------------------------------------------------------------
objective (mean - 1.0*std) : 8.468 bits   (baseline 0.375)
mean entropy : 8.887 bits   (baseline 1.092,  max 17.21)
std          : 0.420   min : 8.127
    8.92 bits  |  What shall I do today?
    9.28 bits  |  What's a quick and healthy dinner I can make tonight?
    8.74 bits  |  Give me a 3

In [11]:
# === Resume GCG from current best; target mean entropy >= 13 bits ===
import time

def gcg_resume(start_ids, target_mean=13.0, max_steps=220, search_width=64,
               topk=512, patience=50, seed=1, verbose_every=10):
    torch.manual_seed(seed)
    ids = start_ids.clone()
    best_H = per_query_entropy(ids); best_obj = objective(best_H).item()
    best_ids = ids.clone(); stale = 0; t0 = time.time()
    for step in range(1, max_steps + 1):
        grad = suffix_grad(ids)
        cand_tok = (-grad).topk(topk, dim=1).indices
        cand = ids.repeat(search_width, 1)
        pos  = torch.randint(0, N_OPT, (search_width,), device=DEV)
        pick = torch.randint(0, topk, (search_width,), device=DEV)
        cand[torch.arange(search_width, device=DEV), pos] = cand_tok[pos, pick]
        cand = torch.cat([ids, cand], dim=0)
        H = per_query_entropy(cand); obj = objective(H)
        j = int(obj.argmax()); ids = cand[j:j+1].clone()
        if obj[j].item() > best_obj + 1e-4:
            best_obj = obj[j].item(); best_ids = cand[j:j+1].clone(); best_H = H[:, j:j+1]
            stale = 0
        else:
            stale += 1
        m = best_H.mean().item()
        if step % verbose_every == 0:
            print(f"step {step:3d}  obj={best_obj:.3f}  mean={m:.3f}  "
                  f"std={best_H.std(unbiased=False):.3f}  min={best_H.min():.3f}  "
                  f"stale={stale}  ({(time.time()-t0)/step:.2f}s/step)")
        if m >= target_mean:
            print(f"** reached target mean {m:.3f} bits at step {step} **"); break
        if stale >= patience:
            print(f"** early stop: no improvement for {patience} steps (step {step}) **"); break
    return best_ids, best_obj, best_H

best_ids, best_obj, best_H = gcg_resume(best_ids, target_mean=13.0)

prefix_str = tok.decode(best_ids[0])
print("\n" + "="*70)
print("prefix ids:", best_ids[0].tolist())
print("prefix    :", repr(prefix_str))
print(f"objective={best_obj:.3f}  mean={best_H.mean():.3f}  "
      f"std={best_H.std(unbiased=False):.3f}  min={best_H.min():.3f}  (max {V_max:.2f})")
for q, h in zip(queries, best_H.squeeze(1).tolist()):
    print(f"   {h:5.2f} bits  |  {q}")

step  10  obj=8.864  mean=9.185  std=0.321  min=8.753  stale=2  (5.82s/step)
step  20  obj=10.194  mean=10.465  std=0.271  min=10.148  stale=0  (5.78s/step)
step  30  obj=10.308  mean=10.661  std=0.354  min=10.245  stale=0  (5.78s/step)
step  40  obj=10.545  mean=10.809  std=0.264  min=10.516  stale=6  (5.77s/step)
step  50  obj=10.718  mean=10.916  std=0.198  min=10.729  stale=1  (5.77s/step)
step  60  obj=10.718  mean=10.916  std=0.198  min=10.729  stale=11  (5.77s/step)
step  70  obj=10.718  mean=10.916  std=0.198  min=10.729  stale=21  (5.77s/step)
step  80  obj=10.975  mean=11.368  std=0.393  min=10.637  stale=6  (5.77s/step)
step  90  obj=10.975  mean=11.368  std=0.393  min=10.637  stale=16  (5.77s/step)
step 100  obj=11.175  mean=11.375  std=0.200  min=11.078  stale=3  (5.77s/step)
step 110  obj=11.303  mean=11.629  std=0.326  min=11.075  stale=1  (5.77s/step)
step 120  obj=11.303  mean=11.629  std=0.326  min=11.075  stale=11  (5.77s/step)
step 130  obj=11.401  mean=11.635  std=

In [12]:
# === Escape the plateau: wider search, larger token pool, fresh seed ===
try:
    best_ids, best_obj, best_H = gcg_resume(
        best_ids, target_mean=13.0, max_steps=180,
        search_width=96, topk=1024, patience=70, seed=3, verbose_every=10)
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    print("OOM at width=96 -> retrying at width=64")
    best_ids, best_obj, best_H = gcg_resume(
        best_ids, target_mean=13.0, max_steps=180,
        search_width=64, topk=1024, patience=70, seed=3, verbose_every=10)

prefix_str = tok.decode(best_ids[0])
print("\n" + "="*70)
print("prefix ids:", best_ids[0].tolist())
print("prefix    :", repr(prefix_str))
print(f"objective={best_obj:.3f}  mean={best_H.mean():.3f}  "
      f"std={best_H.std(unbiased=False):.3f}  min={best_H.min():.3f}  (max {V_max:.2f})")
for q, h in zip(queries, best_H.squeeze(1).tolist()):
    print(f"   {h:5.2f} bits  |  {q}")

step  10  obj=12.261  mean=12.364  std=0.103  min=12.221  stale=3  (8.50s/step)
step  20  obj=12.261  mean=12.364  std=0.103  min=12.221  stale=13  (8.49s/step)
step  30  obj=12.261  mean=12.364  std=0.103  min=12.221  stale=23  (8.49s/step)
step  40  obj=12.261  mean=12.364  std=0.103  min=12.221  stale=33  (8.49s/step)


KeyboardInterrupt: 

In [14]:
# === Mount Google Drive for checkpointing (run once) ===
DRIVE_DIR = None
try:
    from google.colab import drive
    import os
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/gcg_entropy'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print('Drive mounted -> checkpoints ->', DRIVE_DIR)
except Exception as e:
    print('Drive not mounted; checkpointing to /content only:', repr(e))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted -> checkpoints -> /content/drive/MyDrive/gcg_entropy


In [ ]:
# === Open-ended auto-escalating GCG =========================================
# No entropy cap. On every plateau: ramp search params AND grow the prefix +1 token.
# Best prefix is checkpointed to /content and to Drive after each stage.
import json

def save_checkpoint(rec):
    with open('/content/gcg_entropy_best.json', 'w') as f:
        json.dump(rec, f, ensure_ascii=False, indent=2)
    if DRIVE_DIR:
        try:
            with open(f'{DRIVE_DIR}/best.json', 'w') as f:
                json.dump(rec, f, ensure_ascii=False, indent=2)
        except Exception as e:
            print('  drive checkpoint failed:', repr(e))

def record(stage, width, topk, steps, bH, bids):
    return dict(stage=stage, length=int(bids.shape[1]), width=width, topk=topk, steps=steps,
                objective=objective(bH).item(), mean=bH.mean().item(),
                std=bH.std(unbiased=False).item(), min=bH.min().item(),
                per_query=bH.squeeze(1).tolist(),
                prefix_ids=bids[0].tolist(), prefix=tok.decode(bids[0]),
                max_possible=V_max)

def autoscale(start_ids, max_stages=12, width=64, topk=512, steps=150,
              patience=60, conv_gain=0.02):
    global N_OPT
    b_ids = start_ids.clone()
    b_H = per_query_entropy(b_ids); b_obj = objective(b_H).item()
    seed = 10; history = []
    for stage in range(1, max_stages + 1):
        prev = b_obj
        print(f"\n##### stage {stage}: len={N_OPT} width={width} topk={topk} "
              f"steps={steps} patience={patience}  (start obj={b_obj:.3f}) #####")
        try:
            ni, no, nH = gcg_resume(b_ids, target_mean=99.0, max_steps=steps,
                                    search_width=width, topk=topk, patience=patience,
                                    seed=seed, verbose_every=20)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); width = max(32, width // 2)
            print(f"  OOM -> back off width to {width}"); continue
        if no > b_obj:
            b_obj, b_ids, b_H = no, ni, nH
        rec = record(stage, width, topk, steps, b_H, b_ids); history.append(rec)
        save_checkpoint(rec)
        gain = b_obj - prev
        print(f"== stage {stage}: len={N_OPT} obj={b_obj:.3f} mean={b_H.mean():.3f} "
              f"min={b_H.min():.3f} gain={gain:.3f}  (ceiling {V_max:.2f}) ==")
        print("CHECKPOINT_JSON " + json.dumps(rec, ensure_ascii=False))
        if gain < conv_gain and stage >= 3:
            print(f"** converged: stage gain {gain:.3f} < {conv_gain} **"); break
        # --- escalate for the next stage ---
        width = min(192, int(round(width * 1.5)))
        topk  = min(V, topk * 2)
        steps = int(round(steps * 1.25))
        seed += 1
        N_OPT += 1                                   # +1 payload length per plateau
        b_ids = torch.cat([b_ids, b_ids[:, -1:]], dim=1)   # extend prefix, seed new slot
        b_H = per_query_entropy(b_ids); b_obj = objective(b_H).item()
        print(f"  -> grew payload to {N_OPT} tokens (obj now {b_obj:.3f})")
    return b_ids, b_obj, b_H, history

best_ids, best_obj, best_H, history = autoscale(best_ids)

final = record('final', None, None, None, best_H, best_ids)
save_checkpoint(final)
print("\n" + "=" * 70)
print(f"FINAL: len={best_ids.shape[1]}  mean={best_H.mean():.3f} bits  "
      f"std={best_H.std(unbiased=False):.3f}  min={best_H.min():.3f}  (ceiling {V_max:.2f})")
print("prefix:", repr(tok.decode(best_ids[0])))
print("CHECKPOINT_JSON " + json.dumps(final, ensure_ascii=False))


##### stage 1: len=15 width=64 topk=512 steps=150 patience=60  (start obj=12.178) #####
step  20  obj=12.252  mean=12.327  std=0.076  min=12.193  stale=19  (5.79s/step)
step  40  obj=12.252  mean=12.327  std=0.076  min=12.193  stale=39  (5.78s/step)
step  60  obj=12.381  mean=12.511  std=0.130  min=12.238  stale=7  (5.78s/step)
step  80  obj=12.381  mean=12.511  std=0.130  min=12.238  stale=27  (5.78s/step)
step 100  obj=12.381  mean=12.511  std=0.130  min=12.238  stale=47  (5.78s/step)
** early stop: no improvement for 60 steps (step 113) **
== stage 1: len=15 obj=12.381 mean=12.511 min=12.238 gain=0.203  (ceiling 17.21) ==
CHECKPOINT_JSON {"stage": 1, "length": 15, "width": 64, "topk": 512, "steps": 150, "objective": 12.380891799926758, "mean": 12.511190414428711, "std": 0.13029862940311432, "min": 12.237908363342285, "per_query": [12.556571960449219, 12.586539268493652, 12.237908363342285, 12.58997631072998, 12.475552558898926, 12.620593070983887], "prefix_ids": [41405, 119109, 130